In [1]:
"""
PIPELINE STAGE: Renewable Electricity Generation Dataset Enrichment, Spatial Coding & Temporal Standardization

INPUT: processed_data/steps/05_Generation_Cleaned.xlsx
OUTPUT: processed_data/steps/06_Generation_Enriched.xlsx
LIBRARIES: os, pandas

1. OBJECTIVE:
    Enrich the cleaned renewable electricity generation dataset by converting temporal
    variables into numerical format, correcting province inconsistencies, assigning
    official Turkish province plate codes, and producing a fully sorted dataset suitable
    for advanced statistical analysis, spatial modeling, and machine learning pipelines.

2. TEMPORAL STANDARDIZATION:
    A. Month Conversion:
       Convert textual month representations into numeric values (1–12).

    B. Supported Formats:
       Handle both English and Turkish month names, including abbreviations:

           january → 1
           feb      → 2
           ocak     → 1
           aralık   → 12

    C. Robust Mapping:
       - Normalize text using lowercase transformation.
       - Strip whitespace before mapping.
       - Convert all results to numeric format.
       - Handle invalid values using coercion.

3. PROVINCE NAME CLEANING:
    A. Typographical Corrections:
       Fix known inconsistencies in province naming:

           KÜTHAHYA → KÜTAHYA
           ELÂZIĞ   → ELAZIĞ

    B. Spatial Standardization:
       Ensure province names follow official Turkish spelling conventions.

4. PROVINCE IDENTIFICATION:
    A. Dynamic Detection:
       Identify the province column using keyword matching:

           "iller"

       after normalization.

    B. Validation:
       Stop execution if no valid province column is found.

5. OFFICIAL PROVINCE PLATE CODE ASSIGNMENT:
    A. Geographic Coding:
       Map each province to its official Turkish plate number (1–81).

    B. Supported Cases:
       Include both standard and alternative names:

           ADANA → 1
           ANKARA → 6
           İSTANBUL → 34
           İZMİR → 35
           URFA → 63
           ADAPAZARI → 54
           İÇEL → 33

    C. Missing Code Detection:
       Identify provinces that cannot be mapped and report them for review.

6. TEMPORAL DATA VALIDATION:
    A. Year Conversion:
       Convert Year column to numeric format for proper sorting.

    B. Data Integrity:
       Coerce invalid values to NaN instead of causing failure.

7. SPATIAL FEATURE ENGINEERING:
    A. New Feature Creation:
       Add a new column:

           Plaka

       representing official geographic identifiers.

    B. Analytical Utility:
       Enable integration with external datasets such as demographics,
       meteorology, emissions, and energy infrastructure data.

8. DATA SORTING & STRUCTURING:
    A. Multi-Level Sorting:
       Sort dataset by:

           1. Year
           2. Month
           3. Plate (Plaka)

       in ascending order.

    B. Chronological Consistency:
       Ensure time-series structure is preserved for panel data analysis.

9. COLUMN RESTRUCTURING:
    A. Column Placement:
       Reorder columns so that:

           Plaka

       appears immediately next to the province column.

    B. Readability Enhancement:
       Improve dataset usability for reporting and visualization.

10. OUTPUT GENERATION:
    A. Final Export:

           processed_data/steps/06_Generation_Enriched.xlsx

    B. Export Settings:
       - Excel format (.xlsx)
       - No index column

11. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:
           - File processing status
           - Month conversion status
           - Province correction status
           - Plate assignment status
           - Sorting confirmation

    B. Warning Messages:
       Report:
           - Missing province column
           - Provinces without plate mapping
           - Structural inconsistencies

    C. Completion Message:
       Confirm successful generation of enriched dataset.

12. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Province-level renewable electricity generation values
        - Standardized province names
        - Year (numeric)
        - Month (numeric)
        - Official province plate codes (1–81)

    This dataset is fully structured for:
        - Time-series analysis
        - Spatial modeling
        - Machine learning pipelines
        - Energy transition studies
        - Multivariate statistical analysis
"""

import os
import pandas as pd

# 1. Dosya Yolları
BASE_DIR = r"C:\Users\w11\dergi2"
input_file = os.path.join(BASE_DIR, "processed_data", "steps", "05_generation_cleaned.xlsx")
# Yeni dosyayı bir sonraki adım olarak kaydedelim
output_file = os.path.join(BASE_DIR, "processed_data", "steps", "06_generation_enriched.xlsx")

# --- SÖZLÜKLER ---

# Ay isimlerini numaraya çevirme sözlüğü (Küçük harfle yazıldı, veriyi eşleştirirken lower() kullanacağız)
month_map = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
    "ocak": 1, "şubat": 2, "subat": 2, "mart": 3, "nisan": 4, "mayıs": 5, "mayis": 5, "haziran": 6,
    "temmuz": 7, "ağustos": 8, "agustos": 8, "eylül": 9, "eylul": 9, "ekim": 10, "kasım": 11, "kasim": 11, "aralık": 12, "aralik": 12
}

# 81 İlin Plaka Sözlüğü
plaka_map = {
    'ADANA': 1, 'ADIYAMAN': 2, 'AFYONKARAHİSAR': 3, 'AĞRI': 4, 'AMASYA': 5, 'ANKARA': 6, 'ANTALYA': 7, 'ARTVİN': 8, 'AYDIN': 9, 'BALIKESİR': 10,
    'BİLECİK': 11, 'BİNGÖL': 12, 'BİTLİS': 13, 'BOLU': 14, 'BURDUR': 15, 'BURSA': 16, 'ÇANAKKALE': 17, 'ÇANKIRI': 18, 'ÇORUM': 19, 'DENİZLİ': 20,
    'DİYARBAKIR': 21, 'EDİRNE': 22, 'ELAZIĞ': 23, 'ERZİNCAN': 24, 'ERZURUM': 25, 'ESKİŞEHİR': 26, 'GAZİANTEP': 27, 'GİRESUN': 28, 'GÜMÜŞHANE': 29, 'HAKKARİ': 30,
    'HATAY': 31, 'İSKENDERUN': 31, 'ANTAKYA': 31, 'ISPARTA': 32, 'MERSİN': 33, 'İÇEL': 33, 'İSTANBUL': 34, 'İZMİR': 35, 'KARS': 36, 'KASTAMONU': 37, 'KAYSERİ': 38,
    'KIRKLARELİ': 39, 'KIRŞEHİR': 40, 'KOCAELİ': 41, 'KONYA': 42, 'KÜTAHYA': 43, 'MALATYA': 44, 'MANİSA': 45, 'KAHRAMANMARAŞ': 46, 'MARDİN': 47, 'MUĞLA': 48,
    'MUŞ': 49, 'NEVŞEHİR': 50, 'NİĞDE': 51, 'ORDU': 52, 'RİZE': 53, 'SAKARYA': 54, 'ADAPAZARI': 54, 'SAMSUN': 55, 'SİİRT': 56, 'SİNOP': 57, 'SİVAS': 58, 'TEKİRDAĞ': 59,
    'TOKAT': 60, 'TRABZON': 61, 'TUNCELİ': 62, 'ŞANLIURFA': 63, 'URFA': 63, 'UŞAK': 64, 'VAN': 65, 'YOZGAT': 66, 'ZONGULDAK': 67, 'AKSARAY': 68, 'BAYBURT': 69,
    'KARAMAN': 70, 'KIRIKKALE': 71, 'BATMAN': 72, 'ŞIRNAK': 73, 'BARTIN': 74, 'ARDAHAN': 75, 'IĞDIR': 76, 'YALOVA': 77, 'KARABÜK': 78, 'KİLİS': 79,
    'OSMANİYE': 80, 'DÜZCE': 81
}

fix_map = {
    "KÜTHAHYA": "KÜTAHYA",
    "ELÂZIĞ": "ELAZIĞ"
}

def clean_province(text):
    if pd.isna(text):
        return text
   # typo fix
    text = fix_map.get(text, text)

    return text

def enrich_and_sort_data(input_path, output_path):
    print(f">>> İşleniyor: {os.path.basename(input_path)}")
    df = pd.read_excel(input_path)

    # Hangi sütunun iller olduğunu bul
    iller_col = None
    for col in df.columns:
        if "iller" in str(col).strip().lower().replace("i̇", "i"):
            iller_col = col
            break

    if not iller_col:
        print(" [!] Hata: 'İLLER' sütunu bulunamadı!")
        return

    # --- 1. Ay İsimlerini Numaraya Çevir ---
    if "Ay" in df.columns:
        # Önce string yap, küçük harfe çevirip boşlukları al, sonra sözlükten karşılığını bul
        df["Ay"] = df["Ay"].apply(lambda x: month_map.get(str(x).strip().lower(), x))
        # Nümerik değere zorla
        df["Ay"] = pd.to_numeric(df["Ay"], errors='coerce')
        df[iller_col] = df[iller_col].apply(clean_province)
        print(" [✓] 1. Aylar numaraya çevrildi.")
    else:
        print(" [!] 'Ay' sütunu bulunamadı.")

    # --- 3 & 4. Plaka Sütunu Ekle ve Değerleri Yazdır ---
    # İller sütunundaki değere bakıp plaka kodunu getiriyoruz
    df["Plaka"] = df[iller_col].map(plaka_map)
    print(" [✓] 3 & 4. Plaka sütunu eklendi ve kodlar dolduruldu.")

    # Plakası bulunamayan il var mı kontrol edelim (Yazım hatası vb. tespiti için)
    eksik_plakalar = df[df["Plaka"].isna()][iller_col].unique()
    if len(eksik_plakalar) > 0:
        print(f" [!] Uyarı: Plakası bulunamayan veriler var: {eksik_plakalar}")

    # --- 5. Yıl, Ay ve Plaka Bazında Artan Sıralama ---
    # Eğer Yıl sütunu string ise onu da sayıya çevirelim ki doğru sıralansın
    if "Yıl" in df.columns:
        df["Yıl"] = pd.to_numeric(df["Yıl"], errors='coerce')

    # Sıralamayı yap
    df = df.sort_values(by=["Yıl", "Ay", "Plaka"], ascending=[True, True, True])
    print(" [✓] 5. Veriler Yıl, Ay ve Plaka'ya göre artan (ascending) şekilde sıralandı.")

    # Sütun sırasını düzenlemek istersen (İsteğe bağlı, tablo daha şık dursun diye Plakayı başa alıyoruz)
    cols = list(df.columns)
    # Plakayı İller sütununun hemen yanına alalım
    if "Plaka" in cols:
        cols.insert(cols.index(iller_col), cols.pop(cols.index("Plaka")))
        df = df[cols]

    # Kaydet
    df.to_excel(output_path, index=False)
    print(f"\n✅ İŞLEM TAMAM! Yeni dosya kaydedildi: {output_path}")

# --- ÇALIŞTIR ---
if __name__ == "__main__":
    if os.path.exists(input_file):
        enrich_and_sort_data(input_file, output_file)
    else:
        print(f" [!] Dosya bulunamadı: {input_file}")

>>> İşleniyor: 05_generation_cleaned.xlsx
 [✓] 1. Aylar numaraya çevrildi.
 [✓] 3 & 4. Plaka sütunu eklendi ve kodlar dolduruldu.
 [✓] 5. Veriler Yıl, Ay ve Plaka'ya göre artan (ascending) şekilde sıralandı.

✅ İŞLEM TAMAM! Yeni dosya kaydedildi: C:\Users\w11\dergi2\processed_data\steps\06_generation_enriched.xlsx
